In [1]:
import pandas as pd
import numpy as np
import sys
!{sys.executable} -m pip install openpyxl
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font

# Load and clean the data
df = pd.read_csv('cardio_train.csv', sep=';')

# Data cleaning steps
df['age'] = df['age'] / 365.25
df['ap_hi'] = df['ap_hi'].clip(40, 250)
df['ap_lo'] = df['ap_lo'].clip(30, 200)
df['weight'] = df['weight'].clip(30, 200)
df['height'] = df['height'].clip(120, 220)
df['gender'] = df['gender'].map({1: 'female', 2: 'male'})
df['cholesterol'] = df['cholesterol'].map({1: 'normal', 2: 'above normal', 3: 'well above normal'})
df['gluc'] = df['gluc'].map({1: 'normal', 2: 'above normal', 3: 'well above normal'})
binary_cols = ['smoke', 'alco', 'active', 'cardio']
for col in binary_cols:
    df[col] = df[col].astype(bool)
df['bmi'] = df['weight'] / (df['height']/100)**2
bins = [20, 30, 40, 50, 60, 70, 80]
labels = ['20-29', '30-39', '40-49', '50-59', '60-69', '70+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)
conditions = [
    (df['ap_hi'] < 120) & (df['ap_lo'] < 80),
    (df['ap_hi'] < 130) & (df['ap_lo'] < 85),
    (df['ap_hi'] < 140) & (df['ap_lo'] < 90),
    (df['ap_hi'] >= 140) | (df['ap_lo'] >= 90)
]
bp_labels = ['normal', 'elevated', 'stage 1 hypertension', 'stage 2 hypertension']
df['bp_category'] = np.select(conditions, bp_labels)

# Create analysis dataframes
age_group_analysis = df.groupby('age_group')['cardio'].mean().reset_index()
age_group_analysis.columns = ['Age Group', 'Cardiovascular Disease Prevalence']

gender_analysis = df.groupby('gender').agg({
    'cardio': 'mean',
    'smoke': 'mean',
    'alco': 'mean',
    'bmi': 'mean',
    'cholesterol': lambda x: (x != 'normal').mean()
}).reset_index()

bp_distribution = df['bp_category'].value_counts(normalize=True).reset_index()
bp_distribution.columns = ['Blood Pressure Category', 'Percentage']

chol_analysis = df.groupby('cholesterol')['cardio'].mean().reset_index()
chol_analysis.columns = ['Cholesterol Level', 'Cardiovascular Disease Prevalence']

# Create Excel file with single sheet
wb = Workbook()
ws = wb.active
ws.title = "Combined Cleaned Data & Analysis"

# Write cleaned data
ws.append(["CLEANED DATASET"])
for r in dataframe_to_rows(df, index=False, header=True):
    ws.append(r)

# Style headers
bold_font = Font(bold=True)
for row in ws.iter_rows(min_row=1, max_row=1):
    for cell in row:
        cell.font = bold_font

for row in ws.iter_rows():
    for cell in row:
        if cell.value in ["CLEANED DATASET"]:
            cell.font = bold_font

# Save the Excel file
wb.save("combined_cleaned_cardio_data.csv")

C:\Users\hadya\AppData\Local\Temp\ipykernel_20156\3675038781.py:38: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  age_group_analysis = df.groupby('age_group')['cardio'].mean().reset_index()
C:\Users\hadya\anaconda3\envs\ie0005\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")
